# Merging States

The parsers generated by `Larkita` are *canonical* `LR(1)` parsers.  Canonical `LR(1)` automata often have many
more states than `LALR(1)` automata, because they keep apart states that only differ in the look-ahead sets of
their marked rules.  Such states have the same *core*.  With the keyword argument `merge_states=True`, `Larkita`
merges states with the same core, as long as this is safe:

1. Merging must not cause a new conflict, neither in the merged states nor in their successors.
2. Merging must not change how the contextual scanner splits the input into tokens.

Then the resulting parser accepts the same strings and returns the same abstract syntax trees as the canonical
parser.

## A Grammar for Arithmetic Expressions

We use the grammar from the notebook [02-Conflicts-Resolved.ipynb](02-Conflicts-Resolved.ipynb).

In [ ]:
from IPython.display import Code

Code(filename='conflicts-resolved.lark', language='larkita')

In [ ]:
from larkita import Larkita

We generate a canonical parser and a parser with merged states and compare the numbers of their states.

In [ ]:
file_name = 'conflicts-resolved.lark'
canonical = Larkita.open(file_name, out_file='arith.out')
merged    = Larkita.open(file_name, out_file='arith-merged.out',
                         merge_states=True)
print('canonical:', len(canonical.parse_table.states), 'states')
print('merged:   ', len(merged.parse_table.states), 'states')

Both parsers return the same abstract syntax trees.

In [ ]:
for s in ['1+2*3^4', '2^3^4*5+6-7/8^9', '(1+2)*3']:
    assert canonical.parse(s) == merged.parse(s)
    print(merged.parse(s))

The output file of a parser with merged states contains the section `Merged states`.  It explains which states
with the same core have been kept apart.  The function `show_merged_states` prints this section.

In [ ]:
def show_merged_states(out_file):
    with open(out_file) as handle:
        report = handle.read()
    print(report[report.index('Merged states'):report.index('\nStates\n======')].strip())

In [ ]:
show_merged_states('arith-merged.out')

For this grammar, all states with the same core have been merged.  Hence, the merged automaton is the `LALR(1)`
automaton.  Every merged state lists the states of the canonical automaton that it has been merged from, and the
look-ahead set of every marked rule is the union of the look-ahead sets of this rule in these states.  The function
`show_state` prints the states that contain a given marked rule.

In [ ]:
def show_state(out_file, marked_rule):
    with open(out_file) as handle:
        report = handle.read()
    for state in report.split('\n\n\nState ')[1:]:
        if marked_rule in state:
            print('State ' + state.strip() + '\n')

In [ ]:
show_state('arith-merged.out', 'atom → NUMBER •')

In the canonical automaton, the marked rule `atom → NUMBER •` occurs in two states:  The look-ahead set of the first
state contains `$END`, while the look-ahead set of the second state contains `")"`, because the second state is
reached inside parentheses.

In [ ]:
show_state('arith.out', 'atom → NUMBER •')

## The Grammar with the Mysterious Conflict

The grammar from the notebook [05-Mysterious-Conflicts.ipynb](05-Mysterious-Conflicts.ipynb) is an `LR(1)`
grammar, but it is not an `LALR(1)` grammar:  Merging the two states that contain the marked rule `a → X •`
would cause two reduce/reduce conflicts.  Therefore, `Larkita` keeps these states apart.

In [ ]:
Code(filename='mysterious.lark', language='larkita')

In [ ]:
parser = Larkita.open('mysterious.lark', out_file='mysterious-merged.out',
                      merge_states=True)
show_merged_states('mysterious-merged.out')

All four strings of the language are still parsed correctly.

In [ ]:
for s in ['vxy', 'vxz', 'wxy', 'wxz']:
    print(s, parser.parse(s))